# SCD tipo 2

## Célula 1 — Configuração

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
from datetime import datetime

GOLD_PATH = "s3a://gold/tse"

print(f"Gold Path: {GOLD_PATH}")

Gold Path: s3a://gold/tse


## Célula 2 — Carregar dimensão

In [2]:
dim_partido = spark.read.format("delta").load(
    f"{GOLD_PATH}/dim_partido"
)

print("Dimensão carregada.")
dim_partido.show(5, truncate=False)

26/06/23 03:43:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Dimensão carregada.


26/06/23 03:43:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/23 03:43:24 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------------+-------------+-----------------------------+
|numero_partido|sigla_partido|nome_partido                 |
+--------------+-------------+-----------------------------+
|40            |PSB          |PARTIDO SOCIALISTA BRASILEIRO|
|27            |DC           |DEMOCRACIA CRISTÃ            |
|21            |PCB          |PARTIDO COMUNISTA BRASILEIRO |
|22            |PL           |PARTIDO LIBERAL              |
|44            |UNIÃO        |UNIÃO BRASIL                 |
+--------------+-------------+-----------------------------+
only showing top 5 rows



## Célula 3 — Estrutura inicial SCD2

In [3]:
dim_partido_scd2 = (
    dim_partido
    .withColumn("data_inicio", F.current_timestamp())
    .withColumn("data_fim", F.lit(None).cast(TimestampType()))
    .withColumn("registro_ativo", F.lit(True))
)

dim_partido_scd2.show(5, truncate=False)

+--------------+-------------+-----------------------------+--------------------------+--------+--------------+
|numero_partido|sigla_partido|nome_partido                 |data_inicio               |data_fim|registro_ativo|
+--------------+-------------+-----------------------------+--------------------------+--------+--------------+
|40            |PSB          |PARTIDO SOCIALISTA BRASILEIRO|2026-06-23 03:43:24.965534|NULL    |true          |
|27            |DC           |DEMOCRACIA CRISTÃ            |2026-06-23 03:43:24.965534|NULL    |true          |
|21            |PCB          |PARTIDO COMUNISTA BRASILEIRO |2026-06-23 03:43:24.965534|NULL    |true          |
|22            |PL           |PARTIDO LIBERAL              |2026-06-23 03:43:24.965534|NULL    |true          |
|44            |UNIÃO        |UNIÃO BRASIL                 |2026-06-23 03:43:24.965534|NULL    |true          |
+--------------+-------------+-----------------------------+--------------------------+--------+--------

## Célula 4 — Salvar primeira versão

In [4]:
(
    dim_partido_scd2
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/dim_partido_scd2")
)

print("Versão inicial salva.")

Versão inicial salva.


## Célula 5 — Simular nova carga com alteração

In [5]:
nova_carga = (
    dim_partido
    .withColumn(
        "nome_partido",
        F.when(
            F.col("numero_partido") == 15,
            "MDB ALTERADO"
        ).otherwise(F.col("nome_partido"))
    )
)

nova_carga.show(5, truncate=False)

+--------------+-------------+-----------------------------+
|numero_partido|sigla_partido|nome_partido                 |
+--------------+-------------+-----------------------------+
|40            |PSB          |PARTIDO SOCIALISTA BRASILEIRO|
|27            |DC           |DEMOCRACIA CRISTÃ            |
|21            |PCB          |PARTIDO COMUNISTA BRASILEIRO |
|22            |PL           |PARTIDO LIBERAL              |
|44            |UNIÃO        |UNIÃO BRASIL                 |
+--------------+-------------+-----------------------------+
only showing top 5 rows



## Célula 6 — Carregar histórico atual

In [6]:
historico = spark.read.format("delta").load(
    f"{GOLD_PATH}/dim_partido_scd2"
)

historico.show(5, truncate=False)

+--------------+-------------+-----------------------------+-------------------------+--------+--------------+
|numero_partido|sigla_partido|nome_partido                 |data_inicio              |data_fim|registro_ativo|
+--------------+-------------+-----------------------------+-------------------------+--------+--------------+
|40            |PSB          |PARTIDO SOCIALISTA BRASILEIRO|2026-06-23 03:43:25.58269|NULL    |true          |
|27            |DC           |DEMOCRACIA CRISTÃ            |2026-06-23 03:43:25.58269|NULL    |true          |
|21            |PCB          |PARTIDO COMUNISTA BRASILEIRO |2026-06-23 03:43:25.58269|NULL    |true          |
|22            |PL           |PARTIDO LIBERAL              |2026-06-23 03:43:25.58269|NULL    |true          |
|44            |UNIÃO        |UNIÃO BRASIL                 |2026-06-23 03:43:25.58269|NULL    |true          |
+--------------+-------------+-----------------------------+-------------------------+--------+--------------+
o

## Célula 7 — Detectar alteraçõe

In [7]:
alterados = (
    historico.alias("h")
    .filter(F.col("registro_ativo") == True)
    .join(
        nova_carga.alias("n"),
        on="numero_partido"
    )
    .filter(
        F.col("h.nome_partido") != F.col("n.nome_partido")
    )
    .select("numero_partido")
)

alterados.show()

+--------------+
|numero_partido|
+--------------+
|            15|
|            25|
|            25|
+--------------+



## Célula 8 — Encerrar registros antigos

In [8]:
historico_encerrado = (
    historico.alias("h")
    .join(
        alterados.alias("a"),
        on="numero_partido",
        how="left"
    )
    .withColumn(
        "registro_ativo",
        F.when(
            F.col("a.numero_partido").isNotNull(),
            F.lit(False)
        ).otherwise(F.col("registro_ativo"))
    )
    .withColumn(
        "data_fim",
        F.when(
            F.col("registro_ativo") == False,
            F.current_timestamp()
        ).otherwise(F.col("data_fim"))
    )
)

historico_encerrado.show(5, truncate=False)

+--------------+-------------+-----------------------------+-------------------------+--------+--------------+
|numero_partido|sigla_partido|nome_partido                 |data_inicio              |data_fim|registro_ativo|
+--------------+-------------+-----------------------------+-------------------------+--------+--------------+
|40            |PSB          |PARTIDO SOCIALISTA BRASILEIRO|2026-06-23 03:43:25.58269|NULL    |true          |
|27            |DC           |DEMOCRACIA CRISTÃ            |2026-06-23 03:43:25.58269|NULL    |true          |
|21            |PCB          |PARTIDO COMUNISTA BRASILEIRO |2026-06-23 03:43:25.58269|NULL    |true          |
|22            |PL           |PARTIDO LIBERAL              |2026-06-23 03:43:25.58269|NULL    |true          |
|44            |UNIÃO        |UNIÃO BRASIL                 |2026-06-23 03:43:25.58269|NULL    |true          |
+--------------+-------------+-----------------------------+-------------------------+--------+--------------+
o

## Célula 9 — Criar novas versões

In [9]:
novos_registros = (
    nova_carga.alias("n")
    .join(
        alterados.alias("a"),
        on="numero_partido"
    )
    .withColumn("data_inicio", F.current_timestamp())
    .withColumn("data_fim", F.lit(None).cast(TimestampType()))
    .withColumn("registro_ativo", F.lit(True))
)

novos_registros.show(5, truncate=False)

+--------------+-------------+-----------------------------+--------------------------+--------+--------------+
|numero_partido|sigla_partido|nome_partido                 |data_inicio               |data_fim|registro_ativo|
+--------------+-------------+-----------------------------+--------------------------+--------+--------------+
|15            |MDB          |MDB ALTERADO                 |2026-06-23 03:43:30.896765|NULL    |true          |
|25            |PRD          |PARTIDO RENOVACAO DEMOCRATICA|2026-06-23 03:43:30.896765|NULL    |true          |
|25            |PRD          |PARTIDO RENOVAÇÃO DEMOCRÁTICA|2026-06-23 03:43:30.896765|NULL    |true          |
|25            |PRD          |PARTIDO RENOVACAO DEMOCRATICA|2026-06-23 03:43:30.896765|NULL    |true          |
|25            |PRD          |PARTIDO RENOVAÇÃO DEMOCRÁTICA|2026-06-23 03:43:30.896765|NULL    |true          |
+--------------+-------------+-----------------------------+--------------------------+--------+--------

## Célula 10 — Consolidar histórico

In [10]:
historico_final = historico_encerrado.unionByName(
    novos_registros
)

historico_final.show(truncate=False)

+--------------+-------------+----------------------------------------------+-------------------------+--------------------------+--------------+
|numero_partido|sigla_partido|nome_partido                                  |data_inicio              |data_fim                  |registro_ativo|
+--------------+-------------+----------------------------------------------+-------------------------+--------------------------+--------------+
|40            |PSB          |PARTIDO SOCIALISTA BRASILEIRO                 |2026-06-23 03:43:25.58269|NULL                      |true          |
|27            |DC           |DEMOCRACIA CRISTÃ                             |2026-06-23 03:43:25.58269|NULL                      |true          |
|21            |PCB          |PARTIDO COMUNISTA BRASILEIRO                  |2026-06-23 03:43:25.58269|NULL                      |true          |
|22            |PL           |PARTIDO LIBERAL                               |2026-06-23 03:43:25.58269|NULL                 

## Célula 11 — Salvar resultado final

In [11]:
(
    historico_final
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/dim_partido_scd2")
)

print("Histórico SCD2 salvo.")

Histórico SCD2 salvo.


## Célula 12 — Validação final

In [12]:
resultado = spark.read.format("delta").load(
    f"{GOLD_PATH}/dim_partido_scd2"
)

resultado.orderBy(
    "numero_partido",
    F.desc("registro_ativo")
).show(50, truncate=False)

+--------------+-------------+----------------------------------------------+--------------------------+--------------------------+--------------+
|numero_partido|sigla_partido|nome_partido                                  |data_inicio               |data_fim                  |registro_ativo|
+--------------+-------------+----------------------------------------------+--------------------------+--------------------------+--------------+
|10            |REPUBLICANOS |REPUBLICANOS                                  |2026-06-23 03:43:25.58269 |NULL                      |true          |
|11            |PP           |PROGRESSISTAS                                 |2026-06-23 03:43:25.58269 |NULL                      |true          |
|12            |PDT          |PARTIDO DEMOCRÁTICO TRABALHISTA               |2026-06-23 03:43:25.58269 |NULL                      |true          |
|13            |PT           |PARTIDO DOS TRABALHADORES                     |2026-06-23 03:43:25.58269 |NULL          